In [5]:
"""
YOLO 비디오 객체 검출 + 세그멘테이션 (배치 처리)

매 프레임 검출 → 클래스 라벨 + bbox / 마스크 그리기

입력: ./test_frames/frame_00000001.jpg, ...
출력: ./bbox_video.mp4  (bbox 모드)
      ./mask_video.mp4  (mask 모드)

설치:
    pip install ultralytics

실행:
    python yolo_video.py
"""

import os
import glob
import numpy as np
import cv2
import time
from tqdm import tqdm
from ultralytics import YOLO

# ━━━━━━━━━━ 설정 ━━━━━━━━━━
FRAMES_DIR    = "./test_frames"
FRAME_PATTERN = "frame_*.jpg"
BBOX_OUTPUT   = "./bbox_video.mp4"
MASK_OUTPUT   = "./mask_video.mp4"
FPS           = 10

MODEL_NAME  = "yolo26s-seg.pt"  # seg 모델이어야 마스크 출력 가능
CONF_THRESH = 0.25
IOU_THRESH  = 0.45
BATCH_SIZE  = 16

# 출력 모드: "bbox", "mask", "both"
DRAW_MODE = "both"


# ━━━━━━━━━━ 클래스별 고정 색상 ━━━━━━━━━━
def get_color(cls_id):
    return (np.random.RandomState(cls_id + 42).rand(3) * 255).astype(int).tolist()


# ━━━━━━━━━━ 렌더링 함수 ━━━━━━━━━━
def render_bbox(frame, r):
    """bbox + 라벨 그리기"""
    h, w = frame.shape[:2]
    if r.boxes is None or len(r.boxes) == 0:
        return frame
    out = frame.copy()
    for box in r.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        label = f"{r.names[cls_id]} {conf:.2f}"
        color = get_color(cls_id)

        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        fs = max(0.4, min(h, w) / 2000)
        (tw, th_t), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, fs, 1)
        cv2.rectangle(out, (x1, max(y1 - th_t - 8, 0)), (x1 + tw + 6, y1), color, -1)
        cv2.putText(out, label, (x1 + 3, max(y1 - 4, th_t + 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def render_mask(frame, r, alpha=0.45):
    """마스크 오버레이 + 외곽선 + 라벨 그리기"""
    h, w = frame.shape[:2]
    if r.masks is None or len(r.masks) == 0:
        return frame
    out = frame.copy().astype(np.float64)

    masks_data = r.masks.data.cpu().numpy()  # (N, mask_h, mask_w)

    for i in range(len(r.boxes)):
        cls_id = int(r.boxes[i].cls[0])
        conf = float(r.boxes[i].conf[0])
        color = np.array(get_color(cls_id), dtype=float)

        # 마스크를 원본 해상도로 리사이즈
        mask = masks_data[i]
        if mask.shape[0] != h or mask.shape[1] != w:
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
        mask_bool = mask > 0.5

        # 마스크 오버레이
        for c in range(3):
            out[:, :, c] = np.where(
                mask_bool,
                out[:, :, c] * (1 - alpha) + color[c] * alpha,
                out[:, :, c],
            )

        # 외곽선
        out_u8 = np.clip(out, 0, 255).astype(np.uint8)
        contours, _ = cv2.findContours(
            mask_bool.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        cv2.drawContours(out_u8, contours, -1, get_color(cls_id), 2)
        out = out_u8.astype(np.float64)

        # 라벨 (bbox 위치에)
        x1, y1, x2, y2 = r.boxes[i].xyxy[0].cpu().numpy().astype(int)
        label = f"{r.names[cls_id]} {conf:.2f}"
        fs = max(0.4, min(h, w) / 2000)
        (tw, th_t), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, fs, 1)
        cv2.rectangle(out_u8, (x1, max(y1 - th_t - 8, 0)), (x1 + tw + 6, y1), get_color(cls_id), -1)
        cv2.putText(out_u8, label, (x1 + 3, max(y1 - 4, th_t + 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), 1, cv2.LINE_AA)
        out = out_u8.astype(np.float64)

    return np.clip(out, 0, 255).astype(np.uint8)


# ━━━━━━━━━━ 메인 ━━━━━━━━━━
def main():
    paths = sorted(glob.glob(os.path.join(FRAMES_DIR, FRAME_PATTERN)))
    if not paths:
        raise FileNotFoundError(f"프레임 없음: {os.path.join(FRAMES_DIR, FRAME_PATTERN)}")

    sample = cv2.imread(paths[0])
    h, w = sample.shape[:2]
    n_frames = len(paths)

    print(f"{'='*50}")
    print(f"  YOLO 비디오 검출 + 세그멘테이션")
    print(f"  모델: {MODEL_NAME}")
    print(f"  출력: {DRAW_MODE}")
    print(f"  프레임: {n_frames}장, {w}x{h}, FPS={FPS}")
    print(f"  배치: {BATCH_SIZE}")
    print(f"{'='*50}")

    # 모델 로드
    print("\n[1/3] 모델 로딩 중...")
    model = YOLO(MODEL_NAME)

    # 배치 추론
    print("[2/3] 배치 검출 중...")
    t0 = time.time()
    all_results = []
    for i in tqdm(range(0, n_frames, BATCH_SIZE), desc="    추론"):
        batch_paths = paths[i : i + BATCH_SIZE]
        results = model(batch_paths, conf=CONF_THRESH, iou=IOU_THRESH, imgsz=512, verbose=False)
        all_results.extend(results)
    det_time = time.time() - t0
    print(f"    검출 완료: {det_time:.1f}초 ({n_frames / det_time:.1f} fps)")

    # 렌더링
    print("[3/3] 렌더링 중...")
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    do_bbox = DRAW_MODE in ("bbox", "both")
    do_mask = DRAW_MODE in ("mask", "both")
    bbox_writer = cv2.VideoWriter(BBOX_OUTPUT, fourcc, FPS, (w, h)) if do_bbox else None
    mask_writer = cv2.VideoWriter(MASK_OUTPUT, fourcc, FPS, (w, h)) if do_mask else None
    t1 = time.time()

    for fi in tqdm(range(n_frames), desc="    렌더링"):
        frame = cv2.imread(paths[fi])
        r = all_results[fi]

        if do_bbox and bbox_writer:
            bbox_writer.write(render_bbox(frame, r))
        if do_mask and mask_writer:
            mask_writer.write(render_mask(frame, r))

    if bbox_writer: bbox_writer.release()
    if mask_writer: mask_writer.release()

    render_time = time.time() - t1
    total = time.time() - t0

    print(f"\n{'='*50}")
    print(f"  완료!")
    print(f"  검출:   {det_time:.1f}초 ({n_frames / det_time:.1f} fps)")
    print(f"  렌더링: {render_time:.1f}초")
    print(f"  총:     {total:.1f}초")
    if do_bbox: print(f"  BBox → {BBOX_OUTPUT}")
    if do_mask: print(f"  마스크 → {MASK_OUTPUT}")
    print(f"{'='*50}")


if __name__ == "__main__":
    main()

  YOLO 비디오 검출 + 세그멘테이션
  모델: yolo26s-seg.pt
  출력: both
  프레임: 206장, 720x1280, FPS=10
  배치: 16

[1/3] 모델 로딩 중...
[2/3] 배치 검출 중...


    추론: 100%|██████████| 13/13 [00:02<00:00,  5.31it/s]


    검출 완료: 2.4초 (84.1 fps)
[3/3] 렌더링 중...


    렌더링: 100%|██████████| 206/206 [00:07<00:00, 28.05it/s]



  완료!
  검출:   2.4초 (84.1 fps)
  렌더링: 7.3초
  총:     9.8초
  BBox → ./bbox_video.mp4
  마스크 → ./mask_video.mp4
